# Signboard Reader: Final Evaluation Report

**Continuation of:** [Signboard Reader – training notebook](https://www.kaggle.com/code/rakshithanagarajugr/signboard-reader-final/notebook)

This notebook only does **evaluation and reporting**. It loads the trained YOLO weights and produces, in one folder (`/kaggle/working/final_results`):

1. Confusion matrix (counts and normalised)
2. Training / validation result curves
3. Precision, Recall, F1 and PR curves
4. Per-class metrics table
5. Example images for **TP, FP, FN (and TN at image level)**
6. A one-page summary and a downloadable ZIP

> **How to use:** add the training notebook's output (or the dataset holding `best.pt`, `results.csv`, `data.yaml`) as *Input* to this notebook, edit the config cell, then **Run All**.

## 0. Config (the only cell you may need to edit)

In [ ]:
import os, glob, random, shutil, json, warnings, subprocess, sys
from pathlib import Path
warnings.filterwarnings("ignore")

# ============ EDIT ONLY THIS CELL ============
WEIGHTS     = None   # e.g. "/kaggle/input/my-run/weights/best.pt"   (None = auto-search)
DATA_YAML   = None   # e.g. "/kaggle/input/my-dataset/data.yaml"     (None = auto-search)
RESULTS_CSV = None   # training log from the training run            (None = auto-search)

VAL_IMAGES  = None   # e.g. "/kaggle/input/my-dataset/images/valid" (None = auto-search; set if auto-search fails)
DATASET_NAME = "merged_dataset"  # only search inside the attached dataset whose name contains this (None = search all)
SPLIT       = "val"  # "val" or "test"
IMGSZ       = 640    # use the same size you trained with
CONF        = 0.25   # confidence threshold used for the TP/FP/FN analysis
IOU_NMS     = 0.6    # NMS IoU
IOU_MATCH   = 0.5    # a prediction counts as correct if IoU >= this AND class matches
N_EXAMPLES  = 9      # images shown per category (TP / FP / FN / TN)
SEED        = 42
OUT         = Path("/kaggle/working/final_results")
# =============================================

OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

def _norm(x):
    return str(x).lower().replace("-", "_").replace(" ", "_")

try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    import ultralytics
print("ultralytics", ultralytics.__version__)

## 1. Locate weights, dataset and training log

In [ ]:
def find_files(patterns, roots=("/kaggle/working", "/kaggle/input", ".")):
    hits = []
    for r in roots:
        for p in patterns:
            hits += glob.glob(os.path.join(r, "**", p), recursive=True)
    hits = [h for h in hits if "final_results" not in h]
    return sorted(set(hits), key=os.path.getmtime, reverse=True)

if WEIGHTS is None:
    c = [h for h in find_files(["best.pt"])] or find_files(["*.pt"])
    assert c, "No weights found. Set WEIGHTS manually in the config cell."
    WEIGHTS = c[0]

if DATA_YAML is None:
    import yaml
    _cands = sorted(find_files(["*.yaml", "*.yml"]), key=lambda h: 0 if (DATASET_NAME and _norm(DATASET_NAME) in _norm(h)) else 1)
    for h in _cands:
        try:
            d = yaml.safe_load(open(h))
            if isinstance(d, dict) and "names" in d and ("val" in d or "test" in d):
                DATA_YAML = h; break
        except Exception:
            pass
    assert DATA_YAML, "No data.yaml found. Set DATA_YAML manually in the config cell."

if RESULTS_CSV is None:
    c = find_files(["results.csv"])
    RESULTS_CSV = c[0] if c else None

print("WEIGHTS     :", WEIGHTS)
print("DATA_YAML   :", DATA_YAML)
print("RESULTS_CSV :", RESULTS_CSV)

In [ ]:
import yaml
cfg = yaml.safe_load(open(DATA_YAML))
yaml_dir = Path(DATA_YAML).parent

root = Path(str(cfg.get("path", ""))) if cfg.get("path") else yaml_dir
if not root.is_absolute() or not root.exists():
    root = yaml_dir

def resolve_split(entry):
    _manual = globals().get("VAL_IMAGES")
    if _manual:
        return Path(_manual)
    p = Path(str(entry))
    cands = [p] if p.is_absolute() else [root / p, yaml_dir / p]
    s_ = str(entry)
    while s_.startswith("../"):            # Roboflow-style "../valid/images"
        s_ = s_[3:]
        cands += [root / s_, yaml_dir / s_]
    for c in cands:
        if c.exists():
            return c.resolve()

    # --- fallback: search for the validation folder, preferring the named dataset ---
    _dsn = globals().get("DATASET_NAME")
    _n = lambda x: str(x).lower().replace("-", "_").replace(" ", "_")
    tail = str(entry).replace("\\", "/").lstrip("./").strip("/")
    parts = tail.split("/")
    ordered = [tail, "/".join(reversed(parts))]
    if SPLIT == "val":
        ordered += ["images/valid", "images/val", "images/validation", "valid/images", "val/images", "validation/images"]
    else:
        ordered += ["images/test", "test/images"]
    IMG_SFX = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    found = []
    for base in ("/kaggle/input", "/kaggle/working"):
        for dp, dn, fn in os.walk(base):
            dpn = dp.replace("\\", "/")
            for rank, v in enumerate(ordered):
                if dpn.endswith("/" + v) and any(Path(f).suffix.lower() in IMG_SFX for f in fn):
                    found.append((rank, Path(dp))); break
    found.sort(key=lambda t: t[0])
    found = [p for _, p in found]
    if _dsn:
        named = [p for p in found if _n(_dsn) in _n(p)]
        if named:
            found = named
        else:
            print(f"WARNING: no validation folder found inside a dataset named '{_dsn}'. "
                  f"Is it attached under + Add Input? Falling back to any dataset.")
    if found:
        print("Auto-found image folder:", found[0])
        for extra in found[1:4]: print("   other candidate:", extra)
        return found[0]

    # nothing found: show what IS available so the right path can be set
    print("Could not find the images. Folders under /kaggle/input containing images:")
    seen = 0
    for dp, dn, fn in os.walk("/kaggle/input"):
        n = sum(Path(f).suffix.lower() in {".jpg", ".jpeg", ".png"} for f in fn)
        if n:
            print(f"  {dp}  ({n} images)"); seen += 1
            if seen >= 15: break
    if seen == 0:
        print("  (none). The dataset with your images is NOT attached. Add it via '+ Add Input' on the right.")
    raise FileNotFoundError(f"Cannot find images for '{entry}'. Attach merged_dataset (+ Add Input) or set VAL_IMAGES in the config cell.")

split_path = resolve_split(cfg[SPLIT])
_lab = Path(str(split_path).replace(os.sep + "images", os.sep + "labels"))
_n_lab = len(list(_lab.rglob("*.txt"))) if _lab.exists() and _lab != split_path else 0
print(f"label folder: {_lab}  ->  {_n_lab} .txt files")
if _n_lab == 0:
    print("WARNING: no label files found next to the images. FN counts and per-class TP/FN will be wrong until labels are available.")
cfg["path"] = str(root)
cfg[SPLIT] = str(split_path)
names = cfg["names"]
names = {i: n for i, n in enumerate(names)} if isinstance(names, list) else {int(k): v for k, v in names.items()}
cfg["names"] = names
DATA_EVAL = OUT / "data_eval.yaml"
yaml.safe_dump(cfg, open(DATA_EVAL, "w"))
print(f"{SPLIT} images folder:", split_path)
print(len(names), "classes")

## 2. Official YOLO validation (confusion matrix, PR/F1 curves, mAP)

In [ ]:
from ultralytics import YOLO
model = YOLO(WEIGHTS)

metrics = model.val(data=str(DATA_EVAL), split=SPLIT, imgsz=IMGSZ,
                    conf=0.001, iou=0.6,          # standard settings for mAP
                    plots=True, project=str(OUT), name="official_val",
                    exist_ok=True, verbose=False)
val_dir = Path(metrics.save_dir)

print(f"Precision     : {metrics.box.mp:.3f}")
print(f"Recall        : {metrics.box.mr:.3f}")
print(f"mAP@0.5       : {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95  : {metrics.box.map:.3f}")

## 3. Confusion matrix
Rows = **true** class, columns = **predicted** class. The extra `background` row/column captures false alarms (true = background) and missed signs (predicted = background).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import matplotlib.colors as mcolors

cm_raw = metrics.confusion_matrix.matrix          # Ultralytics layout: [pred, true]
cm = cm_raw.T.astype(float)                        # -> [true, pred]
labels = [names[i] for i in range(len(names))] + ["background"]

# row-normalise (each row = % of that true class)
row_sum = cm.sum(1, keepdims=True)
cm_norm = np.divide(cm, row_sum, out=np.zeros_like(cm), where=row_sum > 0)

def annot_arr(vals, pct=False):
    out = np.empty(vals.shape, dtype=object)
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            out[i, j] = "" if v == 0 else (f"{v*100:.0f}" if pct else f"{int(v)}")
    return out

# ---- Normalised
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm_norm, annot=annot_arr(cm_norm, pct=True), fmt="", cmap="Blues", vmin=0, vmax=1,
            xticklabels=labels, yticklabels=labels, linewidths=.4, linecolor="#eeeeee",
            cbar_kws={"label": "Fraction of true class"}, ax=ax, annot_kws={"size": 8})
ax.set_xlabel("Predicted class", fontsize=13); ax.set_ylabel("True class", fontsize=13)
ax.set_title(f"Confusion Matrix (normalised, % per true class), {SPLIT} set", fontsize=15)
plt.xticks(rotation=90); plt.yticks(rotation=0); plt.tight_layout()
plt.savefig(OUT / "confusion_matrix_normalized.png", dpi=200); plt.show()

# ---- Counts
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=annot_arr(cm), fmt="", cmap="Blues", norm=mcolors.PowerNorm(gamma=0.4, vmin=0, vmax=cm.max()),
            xticklabels=labels, yticklabels=labels, linewidths=.4, linecolor="#eeeeee",
            cbar_kws={"label": "Count"}, ax=ax, annot_kws={"size": 8})
ax.set_xlabel("Predicted class", fontsize=13); ax.set_ylabel("True class", fontsize=13)
ax.set_title(f"Confusion Matrix (counts), {SPLIT} set", fontsize=15)
plt.xticks(rotation=90); plt.yticks(rotation=0); plt.tight_layout()
plt.savefig(OUT / "confusion_matrix_counts.png", dpi=200); plt.show()

## 4. Training result curves
Losses should fall; validation metrics should rise then flatten. The red marker shows the epoch of the best checkpoint (`best.pt`).

In [ ]:
if RESULTS_CSV and os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV); df.columns = [c.strip() for c in df.columns]
    panels = [("train/box_loss", "Train box loss"), ("train/cls_loss", "Train class loss"), ("train/dfl_loss", "Train DFL loss"),
              ("metrics/precision(B)", "Precision"), ("metrics/recall(B)", "Recall"),
              ("val/box_loss", "Val box loss"), ("val/cls_loss", "Val class loss"), ("val/dfl_loss", "Val DFL loss"),
              ("metrics/mAP50(B)", "mAP@0.5"), ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]
    fitness = 0.1 * df["metrics/mAP50(B)"] + 0.9 * df["metrics/mAP50-95(B)"]
    best_ep = int(fitness.idxmax())
    ep = df["epoch"] if "epoch" in df else np.arange(1, len(df) + 1)

    fig, axes = plt.subplots(2, 5, figsize=(22, 8))
    for ax, (col, title) in zip(axes.ravel(), panels):
        if col not in df: ax.axis("off"); continue
        ax.plot(ep, df[col], "o-", ms=3, lw=1.8, label="result")
        ax.plot(ep, df[col].rolling(3, min_periods=1, center=True).mean(), ":", lw=2, color="tab:orange", label="smoothed")
        ax.plot(ep.iloc[best_ep], df[col].iloc[best_ep], "r*", ms=13, label="best epoch")
        ax.set_title(title); ax.set_xlabel("epoch"); ax.grid(alpha=.3)
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(f"Training results (best epoch = {int(ep.iloc[best_ep])})", fontsize=16)
    plt.tight_layout(); plt.savefig(OUT / "results_curves.png", dpi=200); plt.show()
    print(df.iloc[best_ep][["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].round(4).to_string())
else:
    print("results.csv not found, set RESULTS_CSV in the config cell to plot the training curves.")

## 5. Precision, Recall, F1 and PR curves (from the official validation)

In [ ]:
from IPython.display import Image, display
for f in ["PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png"]:
    src = val_dir / f
    if src.exists():
        shutil.copy(src, OUT / f)
        print(f); display(Image(filename=str(src), width=700))

## 6. Object-level TP / FP / FN analysis
We run the model on every image at `CONF` and compare against the ground-truth labels:

| Term | Meaning |
|---|---|
| **TP** | predicted box overlaps a real sign (IoU ≥ threshold) **and** has the correct class |
| **FP** | predicted box with no matching real sign (wrong class, duplicate, or nothing there) |
| **FN** | real sign the model did not detect correctly |
| **TN** | not defined for boxes (there are infinite empty boxes); we report it at **image level** instead |

In [ ]:
import cv2
from tqdm.auto import tqdm
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(p):
    p = Path(p)
    if p.is_dir():
        return sorted(f for f in p.rglob("*") if f.suffix.lower() in IMG_EXT)
    return sorted(Path(l.strip()) for l in open(p) if l.strip())

def label_path(img):
    s = str(img); key = os.sep + "images" + os.sep
    i = s.rfind(key)
    if i >= 0: s = s[:i] + os.sep + "labels" + os.sep + s[i + len(key):]
    return Path(s).with_suffix(".txt")

def read_gt(img, w, h):
    lp = label_path(img); boxes, cls = [], []
    if lp.exists():
        for line in open(lp):
            v = line.split()
            if len(v) >= 5:
                c, cx, cy, bw, bh = int(float(v[0])), *map(float, v[1:5])
                boxes.append([(cx - bw/2)*w, (cy - bh/2)*h, (cx + bw/2)*w, (cy + bh/2)*h]); cls.append(c)
    return np.array(boxes).reshape(-1, 4), np.array(cls, dtype=int)

def iou_mat(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def match(gtb, gtc, pb, pc, pconf, thr):
    ious = iou_mat(pb, gtb); used = np.zeros(len(gtb), bool); tp, fp = [], []
    for i in np.argsort(-pconf):
        cand = np.where((~used) & (gtc == pc[i]) & (ious[i] >= thr))[0] if len(gtb) else []
        if len(cand):
            j = cand[np.argmax(ious[i][cand])]; used[j] = True; tp.append(i)
        else:
            kind = "wrong class / duplicate" if len(gtb) and (ious[i] >= thr).any() else "no sign there"
            fp.append((i, kind))
    return tp, fp, np.where(~used)[0]

images = list_images(split_path if split_path.is_dir() else cfg[SPLIT])
print(len(images), "images in", SPLIT)

records = []
B = 16
for k in tqdm(range(0, len(images), B)):
    chunk = images[k:k + B]
    res = model.predict([str(p) for p in chunk], conf=CONF, iou=IOU_NMS, imgsz=IMGSZ, verbose=False)
    for p, r in zip(chunk, res):
        h, w = r.orig_shape
        gtb, gtc = read_gt(p, w, h)
        pb = r.boxes.xyxy.cpu().numpy(); pc = r.boxes.cls.cpu().numpy().astype(int); pf = r.boxes.conf.cpu().numpy()
        tp, fp, fn = match(gtb, gtc, pb, pc, pf, IOU_MATCH)
        records.append(dict(path=str(p), n_gt=len(gtb), n_pred=len(pb),
            tp=[(pb[i], pc[i], pf[i]) for i in tp],
            fp=[(pb[i], pc[i], pf[i], kind) for i, kind in fp],
            fn=[(gtb[j], gtc[j]) for j in fn]))

TP = sum(len(r["tp"]) for r in records); FP = sum(len(r["fp"]) for r in records); FN = sum(len(r["fn"]) for r in records)
prec = TP / max(TP + FP, 1); rec = TP / max(TP + FN, 1); f1 = 2 * prec * rec / max(prec + rec, 1e-9)
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}   (conf>={CONF}, IoU>={IOU_MATCH})")

### 6.1 Per-class table

In [ ]:
rows = []
for c in range(len(names)):
    tp = sum(1 for r in records for t in r["tp"] if t[1] == c)
    fp = sum(1 for r in records for t in r["fp"] if t[1] == c)
    fn = sum(1 for r in records for t in r["fn"] if t[1] == c)
    rows.append(dict(class_name=names[c], support=tp + fn, TP=tp, FP=fp, FN=fn,
                     precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1)))
cls_df = pd.DataFrame(rows)
cls_df["F1"] = 2 * cls_df.precision * cls_df.recall / (cls_df.precision + cls_df.recall).replace(0, np.nan)

# merge official mAP numbers
ap = {}
for i, ci in enumerate(metrics.box.ap_class_index):
    p_, r_, ap50, ap5095 = metrics.box.class_result(i)
    ap[names[int(ci)]] = (ap50, ap5095)
cls_df["mAP50"] = cls_df.class_name.map(lambda n: ap.get(n, (np.nan, np.nan))[0])
cls_df["mAP50-95"] = cls_df.class_name.map(lambda n: ap.get(n, (np.nan, np.nan))[1])
cls_df = cls_df.round(3)
cls_df.to_csv(OUT / "per_class_metrics.csv", index=False)
display(cls_df.style.background_gradient(subset=["precision", "recall", "F1", "mAP50", "mAP50-95"], cmap="RdYlGn", vmin=0, vmax=1).format(precision=3))

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(cls_df)); w_ = .27
ax.bar(x - w_, cls_df.precision, w_, label="Precision"); ax.bar(x, cls_df.recall, w_, label="Recall"); ax.bar(x + w_, cls_df["mAP50"], w_, label="mAP@0.5")
ax.set_xticks(x); ax.set_xticklabels(cls_df.class_name, rotation=75, ha="right"); ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis="y", alpha=.3)
ax.set_title("Per-class performance"); plt.tight_layout(); plt.savefig(OUT / "per_class_bars.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x, cls_df.TP, label="TP", color="#2ca02c"); ax.bar(x, cls_df.FP, bottom=cls_df.TP, label="FP", color="#d62728"); ax.bar(x, cls_df.FN, bottom=cls_df.TP + cls_df.FP, label="FN", color="#1f77b4")
ax.set_xticks(x); ax.set_xticklabels(cls_df.class_name, rotation=75, ha="right"); ax.legend(); ax.grid(axis="y", alpha=.3)
ax.set_title("TP / FP / FN per class"); plt.tight_layout(); plt.savefig(OUT / "per_class_tp_fp_fn.png", dpi=200); plt.show()

## 7. Visual examples: TP, FP, FN and TN
Box colours: 🟩 **green** = correct detection (TP) · 🟥 **red** = false alarm (FP) · 🟦 **blue** = real sign that was missed (FN).

In [ ]:
from matplotlib.patches import Patch
COL = dict(tp=(0, 190, 0), fp=(230, 30, 30), fn=(0, 110, 255))

def draw(rec):
    img = cv2.cvtColor(cv2.imread(rec["path"]), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]; t = max(2, int(round(min(h, w) / 280))); fs = max(.5, min(h, w) / 1000); th_ = max(1, t - 1)
    def box(b, color, text):
        x1, y1, x2, y2 = map(int, b); cv2.rectangle(img, (x1, y1), (x2, y2), color, t)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fs, th_)
        ty = y1 if y1 - th - 6 > 0 else y1 + th + 6
        cv2.rectangle(img, (x1, ty - th - 4), (x1 + tw + 4, ty + 2), color, -1)
        cv2.putText(img, text, (x1 + 2, ty - 1), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), th_, cv2.LINE_AA)
    for b, c, cf in rec["tp"]: box(b, COL["tp"], f"{names[c]} {cf:.2f}")
    for b, c, cf, _ in rec["fp"]: box(b, COL["fp"], f"FP {names[c]} {cf:.2f}")
    for b, c in rec["fn"]: box(b, COL["fn"], f"MISSED {names[c]}")
    return img

def title_of(rec):
    return f"{Path(rec['path']).name[:30]}\nTP {len(rec['tp'])} | FP {len(rec['fp'])} | FN {len(rec['fn'])}"

def show_grid(recs, title, fname, ncols=3):
    if not recs: print(f"No examples for: {title}"); return
    nrows = int(np.ceil(len(recs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 5 * nrows)); axes = np.atleast_1d(axes).ravel()
    for ax, r in zip(axes, recs): ax.imshow(draw(r)); ax.set_title(title_of(r), fontsize=10); ax.axis("off")
    for ax in axes[len(recs):]: ax.axis("off")
    fig.legend(handles=[Patch(color=np.array(COL[k]) / 255, label=l) for k, l in
               [("tp", "TP (correct)"), ("fp", "FP (false alarm)"), ("fn", "FN (missed)")]], loc="upper right", fontsize=11)
    fig.suptitle(title, fontsize=17, y=1.0); plt.tight_layout()
    plt.savefig(OUT / fname, dpi=170, bbox_inches="tight"); plt.show()

rng = random.Random(SEED)
perfect = [r for r in records if r["tp"] and not r["fp"] and not r["fn"]] or [r for r in records if r["tp"]]
with_fp = sorted([r for r in records if r["fp"]], key=lambda r: -max(f[2] for f in r["fp"]))[:max(N_EXAMPLES * 3, 1)]
with_fn = sorted([r for r in records if r["fn"]], key=lambda r: -len(r["fn"]))[:max(N_EXAMPLES * 3, 1)]
tn_imgs = [r for r in records if r["n_gt"] == 0 and r["n_pred"] == 0]

sel = dict(
    TP=rng.sample(perfect, min(N_EXAMPLES, len(perfect))),
    FP=rng.sample(with_fp, min(N_EXAMPLES, len(with_fp))),
    FN=rng.sample(with_fn, min(N_EXAMPLES, len(with_fn))),
    TN=rng.sample(tn_imgs, min(N_EXAMPLES, len(tn_imgs))),
)
show_grid(sel["TP"], "True Positives: signs detected correctly", "examples_TP.png")
show_grid(sel["FP"], "False Positives: model detected a sign that is not there / wrong class", "examples_FP.png")
show_grid(sel["FN"], "False Negatives: real signs the model missed", "examples_FN.png")
show_grid(sel["TN"], "True Negatives (image level): no sign in image, no detection", "examples_TN.png")

# also save every annotated example as its own file
for k, recs in sel.items():
    d = OUT / "example_images" / k; d.mkdir(parents=True, exist_ok=True)
    for r in recs:
        cv2.imwrite(str(d / Path(r["path"]).name), cv2.cvtColor(draw(r), cv2.COLOR_RGB2BGR))

### 7.1 One-slide summary panel (one example of each) + image-level matrix

In [ ]:
panel = [(k, sel[k][0]) for k in ["TP", "FP", "FN", "TN"] if sel[k]]
fig, axes = plt.subplots(1, len(panel), figsize=(6.5 * len(panel), 6)); axes = np.atleast_1d(axes)
desc = dict(TP="True Positive\n(correct detection)", FP="False Positive\n(false alarm)", FN="False Negative\n(missed sign)", TN="True Negative\n(nothing to detect, nothing found)")
for ax, (k, r) in zip(axes, panel):
    ax.imshow(draw(r)); ax.set_title(desc[k], fontsize=14, fontweight="bold"); ax.axis("off")
plt.tight_layout(); plt.savefig(OUT / "summary_panel_TP_FP_FN_TN.png", dpi=200, bbox_inches="tight"); plt.show()

# image-level 2x2: "does this image contain a sign?" vs "did the model find any sign?"
itp = sum(1 for r in records if r["n_gt"] > 0 and r["n_pred"] > 0)
ifn = sum(1 for r in records if r["n_gt"] > 0 and r["n_pred"] == 0)
ifp = sum(1 for r in records if r["n_gt"] == 0 and r["n_pred"] > 0)
itn = sum(1 for r in records if r["n_gt"] == 0 and r["n_pred"] == 0)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap([[itp, ifn], [ifp, itn]], annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, annot_kws={"size": 16},
            xticklabels=["Model: sign found", "Model: nothing found"], yticklabels=["Image has sign", "Image has no sign"])
ax.set_title("Image-level confusion matrix"); plt.tight_layout(); plt.savefig(OUT / "image_level_confusion.png", dpi=200); plt.show()

## 8. Final summary and download

In [ ]:
summary = {
    "weights": str(WEIGHTS), "split": SPLIT, "images_evaluated": len(records), "imgsz": IMGSZ,
    "official_precision": round(float(metrics.box.mp), 4), "official_recall": round(float(metrics.box.mr), 4),
    "mAP50": round(float(metrics.box.map50), 4), "mAP50_95": round(float(metrics.box.map), 4),
    f"object_TP@conf{CONF}": TP, f"object_FP@conf{CONF}": FP, f"object_FN@conf{CONF}": FN,
    f"precision@conf{CONF}": round(prec, 4), f"recall@conf{CONF}": round(rec, 4), f"F1@conf{CONF}": round(f1, 4),
    "image_level": dict(TP=itp, FP=ifp, FN=ifn, TN=itn),
}
json.dump(summary, open(OUT / "summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

worst = cls_df[cls_df.support > 0].sort_values("recall").head(5)[["class_name", "support", "recall", "precision"]]
print("\nWeakest classes by recall:"); print(worst.to_string(index=False))

zip_path = shutil.make_archive("/kaggle/working/final_results_package", "zip", OUT)
print("\nAll files:"); [print("  ", p.relative_to(OUT)) for p in sorted(OUT.rglob("*")) if p.is_file() and "example_images" not in str(p)]
print("\nDownload everything:", zip_path)